### ЗАДАЧА: Прокат техники для команды

Есть список устройств и список операций аренды.
Нужно обработать их через классы и собрать отчёт по занятости техники и доходу.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Класс `Device` с полями `device_id`, `title`, `device_type`, `_daily_price`, `available`.

2. В `Device` реализовать:
   - `@property daily_price`
   - метод `set_daily_price(value)`
   - метод `to_dict()`.

3. В `Device.set_daily_price(value)`:
   - если цена больше нуля, обновить её и вернуть `True`
   - если цена невалидна, ничего не менять и вернуть `False`.

4. Класс `RentalRegistry`:
   - хранит устройства в словаре `self.devices`
   - хранит успешные аренды в списке `self.rentals`
   - хранит проблемы в списке `self.problems`.

5. В `RentalRegistry` реализовать методы:
   - `add_device(device)`
   - `rent(employee, device_id, days)`
   - `return_device(device_id)`
   - `income_by_type()`
   - `busy_devices()`
   - `build_report()`.

6. При обработке аренды:
   - если устройство не найдено, добавить проблему
   - если число дней невалидно, добавить проблему
   - если устройство уже занято, добавить проблему
   - если всё корректно, сохранить аренду и пометить устройство как занятое.

7. После загрузки данных вывести:
   - количество устройств
   - количество успешных аренд
   - общий доход
   - доход по типам техники
   - список занятых устройств
   - список проблем.


In [ ]:
devices_data = [
    ("D-1", "MacBook Pro", "laptop", 45, True),
    ("D-2", "iPad Air", "tablet", 25, True),
    ("D-3", "Canon R50", "camera", 40, True),
    ("D-2", "Duplicate iPad", "tablet", 30, True)
]

rental_ops = [
    ("Алиса", "D-1", 3),
    ("Боб", "D-2", 2),
    ("Кира", "D-2", 4),
    ("Данил", "D-9", 1),
    ("Ева", "D-3", 0)
]

class Device:
    def __init__(self, device_id, title, device_type, daily_price, available):
        self.device_id = device_id
        self.title = title
        self.device_type = device_type
        self._daily_price = daily_price
        self.available = available

    @property
    def daily_price(self):
        return self._daily_price

    def set_daily_price(self, value):
        if value > 0:
            self._daily_price = value
            return True
        return False

    def to_dict(self):
        return {
            'device_id': self.device_id,
            'title': self.title,
            'device_type': self.device_type,
            'daily_price': self.daily_price,
            'available': self.available
        }



class RentalRegistry:
    def __init__(self):
        self.devices = {}
        self.rentals = []
        self.problems = []

    def add_device(self, device):
        if device.device_id in self.devices:
            self.problems.append((device.device_id, "duplicate device"))
            return False
        self.devices[device.device_id] = device
        return True

    def rent(self, employee, device_id, days):
        if device_id not in self.devices:
            self.problems.append((device_id, "device not found"))
            return False

        device = self.devices[device_id]

        if days <= 0:
            self.problems.append((device_id, f"invalid days: {days}"))
            return False

        if not device.available:
            self.problems.append((device_id, "device already rented"))
            return False

        total_price = device.daily_price * days
        rental = {
            'employee': employee,
            'device_id': device_id,
            'days': days,
            'total_price': total_price
        }
        self.rentals.append(rental)
        device.available = False  # Корректное обращение через публичный атрибут
        return True

    def return_device(self, device_id):
        if device_id not in self.devices:
            self.problems.append((device_id, "device not found"))
            return False

        device = self.devices[device_id]
        if device.available:
            self.problems.append((device_id, "device was not rented"))
            return False

        device.available = True
        return True

    def income_by_type(self):
        income_dict = {}
        for rental in self.rentals:
            device_id = rental['device_id']
            if device_id in self.devices:
                device_type = self.devices[device_id].device_type
                income_dict[device_type] = income_dict.get(device_type, 0) + rental['total_price']
        return income_dict

    def busy_devices(self):
        busy = []
        for _, device in self.devices.items():
            if not device.available:
                busy.append(device.title)
        return busy

    def build_report(self):
        return {
            "total_devices": len(self.devices),
            "total_rentals": len(self.rentals),
            "total_income": sum(rental['total_price'] for rental in self.rentals),
            "income_by_type": self.income_by_type(),
            "busy_devices": self.busy_devices(),
            "problems": self.problems
        }

# Инициализация и выполнение
registry = RentalRegistry()

# Добавляем устройства в реестр
for row in devices_data:
    device = Device(*row)
    registry.add_device(device)

# Обрабатываем операции аренды
for employee, device_id, days in rental_ops:
    registry.rent(employee, device_id, days)

report = registry.build_report()

print("Устройств:", report["total_devices"])
print("Успешных аренд:", report["total_rentals"])
print("Доход:", report["total_income"])
print("Доход по типам:", report["income_by_type"])
print("Занятые устройства:", report["busy_devices"])
print("Проблемы:", report["problems"])
